# MVP – Pipeline de Confiabilidade Industrial (SCANIA Component X) para Suporte à Decisão

Base real de caminhões da SCANIA (desafio IDA 2024): um componente de motor anonimizado, com tempo até o reparo e especificações de cada veículo.

**Fluxo:** Entrada → **Bronze** (validação) → **Silver/Gold** (Kaplan-Meier, Weibull com censura, razão de taxa por especificação) → **Decisão** (risco por veículo e reparos esperados) → Exportação (`saida/`)

### Arquivos de entrada (CSV)
| Arquivo | Colunas |
|---|---|
| `train_tte.csv` | `vehicle_id, length_of_study_time_step, in_study_repair` |
| `train_specifications.csv` | `vehicle_id, Spec_0 … Spec_7` |

Os arquivos de validação, teste e as leituras operacionais **não** são necessários.

## 1. Configuração (local ou nuvem)

In [ ]:
import os, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings("ignore")

# Local:  ENTRADA = "entrada"
# Nuvem:  ENTRADA = "s3://meu-bucket/scania"  (requer: pip install fsspec s3fs | gcsfs | adlfs)
ENTRADA = os.environ.get("ENTRADA", "entrada")
SAIDA   = os.environ.get("SAIDA", "saida")
ALFA             = 0.05   # nível do intervalo de confiança (95%)
MIN_EVENTOS      = 20     # reparos mínimos para uma categoria ser analisada
MIN_VEICULOS     = 200    # veículos mínimos por categoria
HORIZONTE_PASSOS = 50     # janela de previsão, em time steps
P_AJUSTADO       = 0.05   # nível do teste após correção de Bonferroni
os.makedirs(f"{SAIDA}/graficos", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 9})
COR = {"azul": "#1F6FB2", "laranja": "#E07B00", "vermelho": "#C0392B", "verde": "#2E8B57", "cinza": "#7F8C8D", "escuro": "#14202B"}

### Funções comuns (taxa de eventos, Weibull com censura, Kaplan-Meier)

In [ ]:
try: from IPython.display import display
except ImportError: display = print

def ic_poisson(k, expo, alfa=ALFA):
    """IC exato (Garwood) da taxa de eventos por unidade de exposição."""
    k = np.asarray(k, float)
    lo = np.where(k > 0, stats.chi2.ppf(alfa/2, 2*k) / 2, 0.0)
    hi = stats.chi2.ppf(1 - alfa/2, 2*k + 2) / 2
    return lo / expo, hi / expo

def weibull_mle(t, d):
    """MLE de Weibull 2p com censura à direita (d=1 reparo, d=0 censurado). Retorna (beta, eta)."""
    from scipy.optimize import minimize
    t, d = np.asarray(t, float), np.asarray(d, float)
    def nll(p):
        b, e = np.exp(p); z = (t/e)**b
        return -(d*(np.log(b) - b*np.log(e) + (b-1)*np.log(t)) - z).sum()
    r = minimize(nll, [0.0, np.log(t.mean())], method="Nelder-Mead", options={"xatol":1e-6, "fatol":1e-8, "maxiter":4000})
    return tuple(np.exp(r.x))

def km(t, d):
    """Kaplan-Meier: retorna (tempos de evento, sobrevivência)."""
    t, d = np.asarray(t, float), np.asarray(d, int)
    o = np.argsort(t); ts, ds = t[o], d[o]
    ut, ev = np.unique(ts[ds == 1], return_counts=True)
    n_risco = len(ts) - np.searchsorted(ts, ut, side="left")
    return ut, np.cumprod(1 - ev / n_risco)

def regime(b):
    return "Desgaste (preventiva por idade)" if b >= 1.3 else ("Falha aleatória (preditiva/condição)" if b > 0.9 else "Mortalidade infantil (revisar instalação)")

## 1b. Subir os 2 CSVs (Colab ou Jupyter)
Rode a célula abaixo e **escolha `train_tte.csv` e `train_specifications.csv`** (pode selecionar os dois de uma vez). No Jupyter, depois de escolher, siga para a próxima célula, que salva os arquivos sozinha.
O notebook reconhece cada arquivo pelo cabeçalho. Se eles já estiverem em `entrada/`, esta etapa é ignorada.

In [ ]:
def cabecalho(caminho):
    try:
        if "://" in caminho:
            import fsspec
            with fsspec.open(caminho, "rt", encoding="utf-8-sig") as f: return f.readline().lower()
        with open(caminho, encoding="utf-8-sig", errors="ignore") as f: return f.readline().lower()
    except Exception: return ""

def detectar():
    """Procura CSVs em ENTRADA (e, se local, na pasta atual, /content e /mnt/data) e classifica pelo cabeçalho."""
    bases = [ENTRADA] if "://" in ENTRADA else [ENTRADA, ".", "/content", "/mnt/data"]
    tte, spec, vistos = [], [], set()
    for base in bases:
        if "://" in base:
            import fsspec; arqs = [o.full_name for o in fsspec.open_files(f"{base}/*.csv")]
        else: arqs = sorted(glob.glob(f"{base}/*.csv"))
        for a in arqs:
            nome = os.path.basename(a)
            if nome in vistos: continue
            h = cabecalho(a)
            if "in_study_repair" in h: tte.append(a); vistos.add(nome)
            elif "spec_0" in h: spec.append(a); vistos.add(nome)
    prefere = lambda L: min(L, key=lambda a: (0 if "train" in os.path.basename(a).lower() else 1, a)) if L else None
    return {"tte": prefere(tte), "spec": prefere(spec)}

up = None
if "://" not in ENTRADA: os.makedirs(ENTRADA, exist_ok=True)

def salvar_uploads():
    # Grava em ENTRADA os arquivos escolhidos no botão de upload (Jupyter)
    v = getattr(up, "value", None) or {}
    itens = v.values() if isinstance(v, dict) else v          # ipywidgets 7 (dict) e 8 (tupla)
    for it in itens:
        with open(f"{ENTRADA}/{it['name']}", "wb") as f: f.write(bytes(it["content"]))
        print("Arquivo salvo:", it["name"])

ARQ = detectar()
faltam = [n for n, k in [("train_tte.csv", "tte"), ("train_specifications.csv", "spec")] if not ARQ[k]]
if faltam and "://" not in ENTRADA:
    print("Ainda não encontrei:", ", ".join(faltam))
    try:                                                       # Google Colab: abre o seletor e salva na hora
        from google.colab import files
        for nome, conteudo in files.upload().items():
            with open(f"{ENTRADA}/{nome}", "wb") as f: f.write(conteudo)
            print("Arquivo salvo:", nome)
    except ImportError:
        try:                                                   # Jupyter / VS Code: botão de upload
            import ipywidgets as widgets
            up = widgets.FileUpload(accept=".csv", multiple=True, description="Escolher CSVs")
            display(up)
        except ImportError:
            print(f"Sem botão de upload neste ambiente. Copie os CSVs para a pasta '{ENTRADA}/' ao lado do notebook.")
else:
    print("Arquivos encontrados - upload ignorado.")

## 2. Detecção dos arquivos

In [ ]:
salvar_uploads()                      # grava o que foi escolhido no botão de upload (se houver)
ARQ = detectar()
assert ARQ["tte"] and ARQ["spec"], (f"Faltam arquivos: preciso de train_tte.csv e train_specifications.csv. "
                                    f"Encontrado: {ARQ}. Pasta atual: {os.getcwd()}. Suba os CSVs na célula 1b ou coloque-os em '{ENTRADA}/'.")
ARQ = {"scania_tte": ARQ["tte"], "scania_spec": ARQ["spec"]}
print("train_tte:", os.path.basename(ARQ["scania_tte"]), "| specifications:", os.path.basename(ARQ["scania_spec"]))

## Análise de sobrevivência com censura
Arquivos: `train_tte.csv` (tempo até o evento: `vehicle_id, length_of_study_time_step, in_study_repair`) e `train_specifications.csv` (`vehicle_id, Spec_0 … Spec_7`).
Veículos sem reparo (`in_study_repair = 0`) são **observações censuradas**: sabemos que sobreviveram até aquele tempo, não quando falharão.
Os tempos são *time steps* relativos e anonimizados (não são horas nem dias).

### 1. Bronze – Ingestão e validação

In [ ]:
tte  = pd.read_csv(ARQ["scania_tte"], encoding="utf-8-sig")
spec = pd.read_csv(ARQ["scania_spec"], encoding="utf-8-sig")
tte.columns, spec.columns = [c.strip() for c in tte.columns], [c.strip() for c in spec.columns]
assert {"vehicle_id","length_of_study_time_step","in_study_repair"} <= set(tte.columns), f"train_tte sem colunas esperadas: {list(tte.columns)}"
SPEC_COLS = [c for c in spec.columns if c.lower().startswith("spec_")]
assert "vehicle_id" in spec.columns and SPEC_COLS, f"train_specifications sem vehicle_id/Spec_*: {list(spec.columns)}"

tte = tte.rename(columns={"length_of_study_time_step": "t", "in_study_repair": "rep"})
tte["t"], tte["rep"] = pd.to_numeric(tte["t"], errors="coerce"), pd.to_numeric(tte["rep"], errors="coerce")
qualidade = dict(veiculos=len(tte), ids_duplicados=int(tte.vehicle_id.duplicated().sum()),
                 tempo_invalido=int((~(tte["t"] > 0)).sum()), rotulo_invalido=int((~tte["rep"].isin([0, 1])).sum()),
                 sem_especificacao=int((~tte.vehicle_id.isin(spec.vehicle_id)).sum()))
print(pd.Series(qualidade))
sc = tte.drop_duplicates("vehicle_id")
sc = sc[(sc["t"] > 0) & sc["rep"].isin([0, 1])].merge(spec.drop_duplicates("vehicle_id"), on="vehicle_id", how="left")
sc["rep"] = sc["rep"].astype(int); sc[SPEC_COLS] = sc[SPEC_COLS].fillna("NA")
print(f"Veículos válidos: {len(sc):,} | reparos: {sc.rep.sum():,} | censurados: {(sc.rep==0).sum():,} | tempo mediano de observação: {sc.t.median():.0f} steps")

### 2. Silver/Gold – Frota: taxa de reparo, Kaplan-Meier e Weibull

In [ ]:
N_EV, EXPO = int(sc.rep.sum()), float(sc.t.sum())
TAXA = N_EV / EXPO                                       # reparos por time step
lo, hi = ic_poisson(N_EV, EXPO)
beta_f, eta_f = weibull_mle(sc.t, sc.rep)
ut, S_km = km(sc.t, sc.rep)
b10 = eta_f * (-np.log(0.9)) ** (1/beta_f)
def S_em(t): i = np.searchsorted(ut, t, side="right") - 1; return 1.0 if i < 0 else S_km[i]
marcos = pd.DataFrame({"t": [100, 200, 300, 400]})
marcos["R_KM_%"] = [100*S_em(t) for t in marcos.t]
marcos["R_Weibull_%"] = [100*np.exp(-(t/eta_f)**beta_f) for t in marcos.t]
print(f"Frota: taxa {1000*TAXA:.3f} reparos por 1.000 steps (IC95% {1000*lo:.3f}–{1000*hi:.3f})")
print(f"Weibull: beta={beta_f:.2f}, eta={eta_f:.0f} steps | vida B10 (10% já reparados): {b10:.0f} steps | regime: {regime(beta_f)}")
display(marcos.round(1))

### 3. Gold – Quais especificações mudam a taxa de reparo? (teste de Poisson com correção de Bonferroni)

In [ ]:
linhas = []
for sp in SPEC_COLS:
    for cat, g in sc.groupby(sp):
        k, ex = int(g.rep.sum()), float(g.t.sum()); esp = TAXA * ex
        l, h = ic_poisson(k, ex)
        linhas.append(dict(spec=sp, categoria=cat, veiculos=len(g), reparos=k, expo=ex, taxa_1000=1000*k/ex,
                           rr=(k/ex)/TAXA, rr_inf=l/TAXA, rr_sup=h/TAXA, esp=esp,
                           p_acima=stats.poisson.sf(k-1, esp), p_abaixo=stats.poisson.cdf(k, esp)))
C = pd.DataFrame(linhas)
C["elegivel"] = (C.veiculos >= MIN_VEICULOS) & (C.reparos >= MIN_EVENTOS)
n_testes = int(C.elegivel.sum())
C["p_ajust"] = np.where(C.elegivel, np.minimum(1, 2 * np.minimum(C.p_acima, C.p_abaixo) * n_testes), np.nan)
def classe(r):
    if not r["elegivel"]: return "Dados insuficientes"
    if r["p_ajust"] < P_AJUSTADO: return "Acima da frota" if r["rr"] > 1 else "Abaixo da frota"
    return "Dentro do esperado"
C["acao"] = C.apply(classe, axis=1)

het = {}
for sp, g in C[C.elegivel].groupby("spec"):
    if len(g) > 1:
        chi = ((g.reparos - g.esp) ** 2 / g.esp).sum(); het[sp] = (stats.chi2.sf(chi, len(g)-1), chi)
SPEC_SEG = min(het, key=lambda s: (het[s][0], -het[s][1]))
print(f"Categorias testadas: {n_testes} | acima da frota: {(C.acao=='Acima da frota').sum()} | abaixo: {(C.acao=='Abaixo da frota').sum()}")
print(f"Especificação que mais separa a taxa de reparo: {SPEC_SEG}")
display(C[C.acao.isin(["Acima da frota","Abaixo da frota"])].sort_values("rr", ascending=False)[["spec","categoria","veiculos","reparos","rr","rr_inf","rr_sup","p_ajust","acao"]].round(3).head(12))

### 4. Weibull por categoria da especificação mais discriminante e risco por veículo

In [ ]:
W = {}
for cat, g in sc.groupby(SPEC_SEG):
    if g.rep.sum() >= MIN_EVENTOS and len(g) >= MIN_VEICULOS: W[cat] = weibull_mle(g.t, g.rep)
tab_w = pd.DataFrame([(SPEC_SEG, c, int((sc[SPEC_SEG]==c).sum()), int(sc[(sc[SPEC_SEG]==c)].rep.sum()), b, e, regime(b))
                      for c, (b, e) in W.items()], columns=["spec","categoria","veiculos","reparos","beta","eta","regime"]).round(2)

H = HORIZONTE_PASSOS
vivos = sc[sc.rep == 0].copy()                           # ainda sem reparo na última observação (censurados)
vivos["beta"] = vivos[SPEC_SEG].map(lambda c: W.get(c, (beta_f, eta_f))[0])
vivos["eta"]  = vivos[SPEC_SEG].map(lambda c: W.get(c, (beta_f, eta_f))[1])
vivos["P_janela_%"] = 100 * (1 - np.exp(-((vivos.t + H) / vivos.eta) ** vivos.beta + (vivos.t / vivos.eta) ** vivos.beta))
risco = vivos.sort_values("P_janela_%", ascending=False).rename(columns={"t": "tempo_obs"})
risco_cat = vivos.groupby(SPEC_SEG).agg(veiculos=("vehicle_id","count"), reparos_esperados=("P_janela_%", lambda s: s.sum()/100),
                                         risco_medio_pct=("P_janela_%","mean")).reset_index().sort_values("reparos_esperados", ascending=False)
print(f"Reparos esperados nos próximos {H} steps entre os {len(vivos):,} veículos em operação: {vivos['P_janela_%'].sum()/100:,.0f}")
display(tab_w)

### 5. Gráficos

In [ ]:
# B5.1 Kaplan-Meier da frota x ajuste Weibull
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.step(ut, 100*S_km, where="post", color=COR["azul"], label="Kaplan-Meier (dados)")
xs = np.linspace(1, sc.t.max(), 300); ax.plot(xs, 100*np.exp(-(xs/eta_f)**beta_f), color=COR["laranja"], lw=2, label=f"Weibull (β={beta_f:.2f}, η={eta_f:.0f})")
ax.set_xlabel("Time steps de operação"); ax.set_ylabel("Sobrevivência R(t) (%)"); ax.set_title("Confiabilidade do Componente X"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{SAIDA}/graficos/scania_01_km_weibull.png"); plt.show()

# B5.2 Kaplan-Meier por categoria da especificação mais discriminante
fig, ax = plt.subplots(figsize=(8, 4.4))
for cat in sc[SPEC_SEG].value_counts().head(5).index:
    g = sc[sc[SPEC_SEG] == cat]; u, s_ = km(g.t, g.rep); ax.step(u, 100*s_, where="post", label=f"{cat} (n={len(g):,})")
ax.set_xlabel("Time steps de operação"); ax.set_ylabel("Sobrevivência (%)"); ax.set_title(f"Sobrevivência por {SPEC_SEG}"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{SAIDA}/graficos/scania_02_km_categorias.png"); plt.show()

# B5.3 Razão de taxa (vs frota) por categoria, com IC 95%
E = C[C.elegivel].sort_values("rr")
if len(E) > 20: E = pd.concat([E.head(10), E.tail(10)])
cor = {"Acima da frota": COR["vermelho"], "Abaixo da frota": COR["verde"], "Dentro do esperado": COR["cinza"]}
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.errorbar(E.rr, range(len(E)), xerr=[E.rr - E.rr_inf, E.rr_sup - E.rr], fmt="none", ecolor=COR["escuro"], capsize=2)
ax.scatter(E.rr, range(len(E)), c=[cor[a] for a in E.acao], zorder=3)
ax.set_yticks(range(len(E))); ax.set_yticklabels([f"{s}: {c}" for s, c in zip(E.spec, E.categoria)], fontsize=8)
ax.axvline(1, ls="--", color="k", lw=1); ax.set_xscale("log"); ax.set_xlabel("Razão de taxa de reparo vs frota (IC 95%)")
ax.set_title("Categorias que mudam a taxa de reparo (vermelho = acima, verde = abaixo)")
plt.tight_layout(); plt.savefig(f"{SAIDA}/graficos/scania_03_razao_taxa.png"); plt.show()

# B5.4 Reparos esperados na janela, por categoria
r = risco_cat.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.barh(r[SPEC_SEG].astype(str), r["reparos_esperados"], color=COR["azul"])
for i, v in enumerate(r["reparos_esperados"]): ax.text(v, i, f" {v:.0f}", va="center", fontsize=8)
ax.set_xlabel(f"Reparos esperados nos próximos {H} steps"); ax.set_title(f"Demanda de reparo por {SPEC_SEG}")
plt.tight_layout(); plt.savefig(f"{SAIDA}/graficos/scania_04_risco_categoria.png"); plt.show()

# B5.5 Distribuição do risco por veículo
p95 = vivos["P_janela_%"].quantile(.95)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.hist(vivos["P_janela_%"], bins=50, color=COR["azul"]); ax.axvline(p95, ls="--", color=COR["vermelho"])
ax.text(p95, ax.get_ylim()[1]*.9, f" 5% mais arriscados (≥ {p95:.1f}%)", color=COR["vermelho"], fontsize=8)
ax.set_xlabel(f"Probabilidade de reparo nos próximos {H} steps (%)"); ax.set_ylabel("Veículos"); ax.set_title("Risco por veículo")
plt.tight_layout(); plt.savefig(f"{SAIDA}/graficos/scania_05_risco_distribuicao.png"); plt.show()

### 6. Exportação (camada Gold para BI)

In [ ]:
C.round(4).to_csv(f"{SAIDA}/scania_categorias.csv", index=False)
tab_w.to_csv(f"{SAIDA}/scania_weibull_categoria.csv", index=False)
risco_cat.round(3).to_csv(f"{SAIDA}/scania_risco_categoria.csv", index=False)
risco[["vehicle_id", "tempo_obs", *SPEC_COLS, "P_janela_%"]].round(3).to_csv(f"{SAIDA}/scania_risco_veiculos.csv", index=False)
res_sc = dict(veiculos=int(len(sc)), reparos=N_EV, censurados=int((sc.rep==0).sum()), tempo_mediano_obs=round(float(sc.t.median()),0),
    taxa_1000_steps=round(1000*TAXA,3), taxa_ic=[round(1000*lo,3), round(1000*hi,3)],
    weibull_frota=dict(beta=round(float(beta_f),2), eta=round(float(eta_f),0), b10=round(float(b10),0), regime=regime(beta_f)),
    marcos=marcos.round(1).to_dict("records"), spec_mais_discriminante=SPEC_SEG, categorias_testadas=n_testes,
    acima_da_frota=C[C.acao=="Acima da frota"].sort_values("rr", ascending=False)[["spec","categoria","veiculos","reparos","rr","rr_inf","rr_sup","p_ajust"]].round(3).head(8).to_dict("records"),
    abaixo_da_frota=C[C.acao=="Abaixo da frota"].sort_values("rr")[["spec","categoria","veiculos","reparos","rr","rr_inf","rr_sup","p_ajust"]].round(3).head(5).to_dict("records"),
    weibull_categorias=tab_w.to_dict("records"), horizonte_steps=H,
    reparos_esperados_janela=round(float(vivos["P_janela_%"].sum()/100),0), veiculos_em_operacao=int(len(vivos)),
    risco_p95_pct=round(float(p95),2), top_categorias_risco=risco_cat.head(5).round(2).to_dict("records"))
json.dump(res_sc, open(f"{SAIDA}/resumo_scania.json", "w"), ensure_ascii=False, indent=2, default=str)
print(json.dumps(res_sc, ensure_ascii=False, indent=2, default=str))

## 7. Salvar os resultados no Google Drive (opcional)
- **Google Colab:** a célula monta o seu Drive (abre uma janela para você autorizar) e copia a pasta `saida/` para `Meu Drive/<PASTA_DRIVE>/`.
- **Jupyter local:** se você usa o Google Drive para computador, informe em `DRIVE_LOCAL` o caminho da pasta sincronizada (ex.: `G:/Meu Drive` ou `~/Google Drive`).
- Para desligar, coloque `SALVAR_NO_DRIVE = False`. Os arquivos continuam sempre em `saida/`.

In [ ]:
SALVAR_NO_DRIVE = True
PASTA_DRIVE = "confiabilidade_scania"                   # pasta criada dentro do seu Drive
DRIVE_LOCAL = os.environ.get("DRIVE_LOCAL")             # fora do Colab: ex. "G:/Meu Drive"

import shutil
def raiz_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")                   # pede autorização na primeira vez
        return "/content/drive/MyDrive"
    except ImportError:
        return os.path.expanduser(DRIVE_LOCAL) if DRIVE_LOCAL else None

if SALVAR_NO_DRIVE and "://" not in SAIDA:
    raiz = raiz_drive()
    if raiz and os.path.isdir(raiz):
        destino = os.path.join(raiz, PASTA_DRIVE)
        shutil.copytree(SAIDA, destino, dirs_exist_ok=True)
        print("Resultados copiados para o Drive:", destino)
    else:
        print("Drive não disponível aqui. No Colab, autorize o acesso; no Jupyter local, preencha DRIVE_LOCAL "
              "com a pasta do Google Drive para computador. Os resultados continuam em", SAIDA)
else:
    print("Envio ao Drive desligado. Resultados em", SAIDA)